# 01 Data Cleaning
**Mục tiêu:** Làm sạch `Credit Risk Data.csv` → xuất `loan_clean.csv` cho EDA & modeling.

**4 vấn đề chất lượng dữ liệu đã phát hiện ở bước Data Understanding:**
1. Giá trị phi lý: `person_age` > 100 (5 dòng), `person_emp_length` > 60 (2 dòng)
2. Thiếu dữ liệu: `loan_int_rate` (~9.6%), `person_emp_length` (~2.7%)
3. Thu nhập cực cao cần điều tra: `person_income` max = 6.000.000 USD
4. Trùng lặp, kiểu dữ liệu

## 1. Đọc dữ liệu & kiểm tra ban đầu

In [13]:
import pandas as pd # thư viện xử lý và phân tích dữ liệu
import numpy as np # thư viện cho ma trận và mảng

df = pd.read_csv(r'D:\Data Personal Project\Credit Risk Analytics\Data\raw\Credit Risk Data.csv')

print('Kích thước dữ liệu', df.shape)   # (số dòng, số cột)
df.head(5)

Kích thước dữ liệu (32581, 29)


,client_ID,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,...,city_latitude,city_longitude,employment_type,loan_term_months,loan_to_income_ratio,other_debt,debt_to_income_ratio,open_accounts,credit_utilization_ratio,past_delinquencies
0,CUST_00001,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,...,43.6532,-79.3832,Self-employed,36,0.593220,8402.453850,0.735635,14,0.495557,0
1,CUST_00002,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,...,43.6532,-79.3832,Full-time,36,0.104167,1607.802794,0.271646,10,0.585436,3
2,CUST_00003,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,...,51.6214,-3.9436,Full-time,36,0.572917,2760.505633,0.860469,14,0.750732,0
3,CUST_00004,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,...,49.2827,-123.1207,Part-time,12,0.534351,7155.286150,0.643592,15,0.379333,0
4,CUST_00005,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,...,42.8864,-78.8784,Part-time,36,0.643382,15626.153440,0.930628,4,0.228103,0


In [14]:
# Tổng quan kiểu dữ liệu 
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   client_ID                   32581 non-null  str    
 1   person_age                  32581 non-null  int64  
 2   person_income               32581 non-null  int64  
 3   person_home_ownership       32581 non-null  str    
 4   person_emp_length           31686 non-null  float64
 5   loan_intent                 32581 non-null  str    
 6   loan_grade                  32581 non-null  str    
 7   loan_amnt                   32581 non-null  int64  
 8   loan_int_rate               29465 non-null  float64
 9   loan_status                 32581 non-null  int64  
 10  loan_percent_income         32581 non-null  float64
 11  cb_person_default_on_file   32581 non-null  str    
 12  cb_person_cred_hist_length  32581 non-null  int64  
 13  gender                      32581 non-null

In [15]:
missing = df.isnull().sum() # Đếm số giá trị thiếu mỗi cột
print(missing[missing > 0])

person_emp_length     895
loan_int_rate        3116
dtype: int64


## 2. Xác nhận lỗi trước khi xóa

In [16]:
# Các dòng tuổi > 100 
print('person_age > 100')
display(df[df['person_age'] > 100][['client_ID','person_age','person_emp_length','person_income']])

# Các dòng số năm làm việc > 60 
print('person_emp_length > 60')
display(df[df['person_emp_length'] > 60][['client_ID','person_age','person_emp_length']])

person_age > 100


,client_ID,person_age,person_emp_length,person_income
81,CUST_00082,144,4.0,250000
183,CUST_00184,144,4.0,200000
575,CUST_00576,123,2.0,80004
747,CUST_00748,123,7.0,78000
32297,CUST_32298,144,12.0,6000000


person_emp_length > 60


,client_ID,person_age,person_emp_length
0,CUST_00001,22,123.0
210,CUST_00211,21,123.0


## 3. Xử lý giá trị phi lý 

In [17]:
rows_before = df.shape[0]

df = df[
    (df['person_age'] <= 100) &
    (df['person_emp_length'].isna() | (df['person_emp_length'] <= 60))
]

print(f'Đã xóa {rows_before - df.shape[0]} dòng có giá trị phi lý')
print('Kích thước dữ liệu sau khi xóa:', df.shape)

Đã xóa 7 dòng có giá trị phi lý
Kích thước dữ liệu sau khi xóa: (32574, 29)


## 4. Xử lý thiếu dữ liệu

Hai cột thiếu — chiến lược điền KHÁC NHAU, dựa trên bản chất từng cột:

### 4a. `loan_int_rate` — điền theo MEDIAN của từng `loan_grade`

**Lý do:** Lãi suất gắn với hạng tín dụng (grade A an toàn → lãi thấp ~7.5%; grade G rủi ro → lãi cao ~20%).                            
Điền một con số chung cho tất cả sẽ bóp méo quan hệ này → Điền theo median (giá trị trung vị) *trong từng grade* để giữ đúng cấu trúc thật do median không bị kéo lệch bởi giá trị outlier

In [18]:
# Kiểm tra lãi suất median theo từng grade 
print(df.groupby('loan_grade')['loan_int_rate'].median().round(2))

loan_grade
A     7.49
B    10.99
C    13.48
D    15.31
E    16.82
F    18.54
G    20.16
Name: loan_int_rate, dtype: float64


In [19]:
# Điền missing loan_int_rate bằng median của chính grade đó
df['loan_int_rate'] = df['loan_int_rate'].fillna( 
    df.groupby('loan_grade')['loan_int_rate'].transform('median') # nhóm dữ liệu theo loan_grade → transform() trả giá trị median tương ứng grade cho các NaN
)

print('loan_int_rate còn thiếu:', df['loan_int_rate'].isnull().sum())

loan_int_rate còn thiếu: 0


### 4b. `person_emp_length` — điền theo MEDIAN tổng thể

**Lý do:** Số năm đi làm không gắn chặt với biến phân loại sẵn có như trường hợp của lãi suất gắn với grade.                     
Chỉ thiếu 2.7% → điền median tổng thể là đủ tốt, đơn giản, không tác động lớn đến cấu trúc dữ liệu

In [20]:
emp_median = df['person_emp_length'].median()
print('Median số năm làm việc:', emp_median)

df['person_emp_length'] = df['person_emp_length'].fillna(emp_median)
print('person_emp_length còn thiếu:', df['person_emp_length'].isnull().sum())

Median số năm làm việc: 4.0
person_emp_length còn thiếu: 0


## 5. Thu nhập cao (person_income)

`max = 6,000,000 USD/năm`

In [21]:
# Xem phân vị cao của thu nhập để đánh giá mức độ "cực đoan"
print(df['person_income'].quantile([0.95, 0.99, 0.999, 1.0]).round(0))

# Đếm số người thu nhập > 1 triệu
print('\nSố người thu nhập > 1,000,000:', (df['person_income'] > 1_000_000).sum())

0.950     138000.0
0.990     225000.0
0.999     641124.0
1.000    2039784.0
Name: person_income, dtype: float64

Số người thu nhập > 1,000,000: 8


**Quyết định:** Giữ nguyên thu nhập cao. Lý do:
- Số lượng ít người có thu nhập cao vượt trội
- Với mô hình tín dụng, người thu nhập cao thường là tín hiệu *tốt* (ít vỡ nợ)

## 6. Kiểm tra trùng lặp 

In [22]:
# Trùng lặp toàn dòng
print('Số dòng trùng lặp:', df.duplicated().sum())

# Trùng client_ID (mỗi khách phải là duy nhất)
print('client_ID trùng:', df['client_ID'].duplicated().sum())

Số dòng trùng lặp: 0
client_ID trùng: 0


In [29]:
remaining = df.isnull().sum()
print('Các cột còn thiếu:')
print(remaining[remaining > 0] if remaining.sum() > 0 else 'KHÔNG còn missing')

Các cột còn thiếu:
KHÔNG còn missing


In [30]:
# Sanity check cuối: min/max các biến số quan trọng đã hợp lý chưa
check_cols = ['person_age','person_emp_length','person_income','loan_int_rate','loan_amnt']
df[check_cols].describe().round(2)

,person_age,person_emp_length,person_income,loan_int_rate,loan_amnt
count,32574.00,32574.00,32574.00,32574.00,32574.00
mean,27.72,4.76,65878.48,11.01,9588.02
std,6.20,3.98,52531.94,3.21,6320.25
min,20.00,0.00,4000.00,5.42,500.00
25%,23.00,2.00,38500.00,7.88,5000.00
50%,26.00,4.00,55000.00,10.99,8000.00
75%,30.00,7.00,79200.00,13.48,12200.00
max,94.00,41.00,2039784.00,23.22,35000.00


## 7. Xuất dữ liệu sạch

Lưu vào `Data/processed/` 

In [34]:
df.to_csv('../Data/processed/credit_risk_data_clean.csv', index=False)
print('Đã lưu')
print('Kích thước dữ liệu sau khi làm sạch:', df.shape)

Đã lưu
Kích thước dữ liệu sau khi làm sạch: (32574, 29)
